In [7]:
# to import mlflow and know the version and trackinig 
import mlflow 
from mlflow import MlflowClient
print("MLflow version:", mlflow.__version__)
print("MLflow tracking URI:",mlflow.get_tracking_uri())

MLflow version: 3.10.1
MLflow tracking URI: sqlite:///c:\Users\PC\codegpt\MLOps\mlflow\mlruns.db


In [8]:
# to store the model in mlflow and use SQLite Backend store and file store for artifacts

from pathlib import Path

PROJECT_DIR =Path.cwd()
DB_PATH = PROJECT_DIR / "mlruns.db"


TRACKING_URI = f"sqlite:///{DB_PATH}"
mlflow.set_tracking_uri(TRACKING_URI)


print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow artifact URI:", mlflow.get_artifact_uri())


 

MLflow tracking URI: sqlite:///c:\Users\PC\codegpt\MLOps\mlflow\mlruns.db
MLflow artifact URI: file:///c:/Users/PC/codegpt/MLOps/mlruns/0/171919a2ef5c49338d68ff2eb83e66c3/artifacts


In [9]:
# to bild the Experiment and set the experiment name and get the experiment id
EXPERIMENT_NAME = "my_experiment"

# set_experiment = create_experiment if not exist
mlflow.set_experiment(EXPERIMENT_NAME)

# get_experiment_by_name
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)


Experiment Name: my_experiment
Experiment ID: 1


In [13]:

# create a smallest useful Mlflow run
mlflow.end_run()  # End any existing run before starting a new one
with mlflow.start_run(run_name="my_run_1") as run:
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", 0.95)
    mlflow.set_tag("tag1", "value1")
  
    print("Run ID:", run.info.run_id)
    print("Run Info:", run.info)

    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    
    
print("Run ID:", run_id)
print("Experiment ID:", experiment_id)
print("Run URI:", mlflow.get_run(run_id))
mlflow.end_run()

Run ID: 577c6399050546c88949833c33398041
Run Info: <RunInfo: artifact_uri='file:///c:/Users/PC/codegpt/MLOps/mlruns/1/577c6399050546c88949833c33398041/artifacts', end_time=None, experiment_id='1', lifecycle_stage='active', run_id='577c6399050546c88949833c33398041', run_name='my_run_1', start_time=1788287341385, status='RUNNING', user_id='PC'>
Run ID: 577c6399050546c88949833c33398041
Experiment ID: 1
Run URI: <Run: data=<RunData: metrics={'accuracy': 0.95}, params={'learning_rate': '0.1'}, tags={'mlflow.runName': 'my_run_1',
 'mlflow.source.name': 'mlflow.ipynb',
 'mlflow.source.type': 'NOTEBOOK',
 'mlflow.user': 'PC',
 'tag1': 'value1'}>, info=<RunInfo: artifact_uri='file:///c:/Users/PC/codegpt/MLOps/mlruns/1/577c6399050546c88949833c33398041/artifacts', end_time=1788287341454, experiment_id='1', lifecycle_stage='active', run_id='577c6399050546c88949833c33398041', run_name='my_run_1', start_time=1788287341385, status='FINISHED', user_id='PC'>, inputs=<RunInputs: dataset_inputs=[], model

In [14]:
# log single and multiple parametrs and metrics
with mlflow.start_run(run_name="my_run_2") as run:
    mlflow.log_param("Algorithm","Random_Forest")
    mlflow.log_params({
        "learing_rate": 0.01,
        "batch_size": 32,
        "max_depth": 5,
        "random_state": 42
    })
    
mlflow.end_run()
    

In [15]:
# log files and structured  artifacts


artifact_dir = Path("mlflow_demo_artifacts")
artifact_dir.mkdir(exist_ok=True)


text_file = artifact_dir / "notes.txt"
text_file.write_text("This file was produced during an MLflow run.\n", encoding="utf-8")

report = {
    "purpose": "MLflow artifact demo",
    "status": "complete",
    "important": True,
}



with mlflow.start_run(run_name="artifacts_demo"):
    mlflow.log_artifact(str(text_file), artifact_path="files")
    mlflow.log_dict(report, "reports/report.json")

    print("Artifacts logged.")
    
mlflow.end_run()


Artifacts logged.


In [16]:
# save a image from matplotlib and log it as an artifact
import matplotlib.pyplot as plt 

with mlflow.start_run(run_name="figure_demo") as run:
    steps = list(range(10))
    loss = [1 / (i + 1) for i in steps]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, loss)
    ax.set_title("Training Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True)

    mlflow.log_figure(fig, "plots/training_loss.png")
    plt.close(fig)
    print("Figure logged as an artifact.")
mlflow.end_run()

Figure logged as an artifact.


In [17]:
# Cell 9 — compare multiple real ML models

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=42,
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=50,random_state=42,),
}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
        })

        mlflow.set_tag("dataset", "sklearn_diabetes")
        mlflow.set_tag("model_name", model_name)

        print(f"{model_name:18s} RMSE={rmse:.3f}, R2={r2:.3f}")

LinearRegression   RMSE=53.853, R2=0.453
RandomForest       RMSE=55.174, R2=0.425


In [18]:
from mlflow.models import infer_signature

model = RandomForestRegressor(n_estimators=50, random_state=42)

with mlflow.start_run(run_name="RandomForestRegressor") as run :
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
     
    signature = infer_signature(X_train,predictions)
    
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model",
        signature=signature,
    )
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    mlflow.log_params(model.get_params())
    mlflow.log_metrics({
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    })

    mlflow.set_tag("dataset", "sklearn_diabetes")
    mlflow.set_tag("model_name", "RandomForestRegressor")
    
    print("\nRun ID:", run.info.run_id)
    print("\nModel URI:", model_info.model_uri)
    print(f"\nRandomForestRegressor RMSE={rmse:.3f}, R2={r2:.3f}")
mlflow.end_run()

2026/09/01 20:29:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 20:29:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Run ID: e3952d24a6424bc6a0f1527c1dbf6354

Model URI: models:/m-c3f1609ff1c847eaac8076125d1a8dfc

RandomForestRegressor RMSE=55.174, R2=0.425


In [ ]:
# try to get experiment from mlflow by mlflowclient

from mlflow import MlflowClient
client = MlflowClient()


experiment_name = "my_experiment"
experiment = client.get_experiment_by_name(experiment_name)
experimernt_id = experiment.experiment_id

run = client.get_run(run_id)

print("Run ID:", run.info.run_id)
print("Experiment ID:", run.info.experiment_id)
print("Status:", run.info.status)

print("Parameters:", run.data.params)
print("Metrics:", run.data.metrics)
print("Tags:", run.data.tags)

Run ID: 577c6399050546c88949833c33398041
Experiment ID: 1
Status: FINISHED
Parameters: {'learning_rate': '0.1'}
Metrics: {'accuracy': 0.95}
Tags: {'mlflow.user': 'PC', 'mlflow.source.name': 'mlflow.ipynb', 'mlflow.source.type': 'NOTEBOOK', 'mlflow.runName': 'my_run_1', 'tag1': 'value1'}


In [ ]:
# use search_runs to get all runs for the experiment and order by RMSE descending


runs = client.search_runs(
    experiment_ids=[experimernt_id],
    order_by=["metrics.rmse DESC"],
)
print(f"\nTop {len(runs)} runs for experiment '{experiment_name}':")
for run in runs:
    rmse = run.data.metrics.get("rmse", float("nan"))
    r2 = run.data.metrics.get("r2", float("nan"))
    print(f"Run ID: {run.info.run_id}, RMSE: {rmse:.3f}, R2: {r2:.3f}")
    


Top 22 runs for experiment 'my_experiment':
Run ID: e3952d24a6424bc6a0f1527c1dbf6354, RMSE: 55.174, R2: 0.425
Run ID: 2e03d2022a2c460697492591504eaced, RMSE: 55.174, R2: 0.425
Run ID: 3ee177a9ef35485bb1687afa7e64c6f5, RMSE: 55.174, R2: 0.425
Run ID: bb91c8f70dc24b43b718259fea53dbdd, RMSE: 55.174, R2: 0.425
Run ID: f6cdcf21303d479e84d8e5e2e2ad642f, RMSE: 55.174, R2: 0.425
Run ID: 65bd7b6493854002ab71b7b77d55dd42, RMSE: 55.174, R2: 0.425
Run ID: c8daa0dd4e7d480abc9ef3653824d170, RMSE: 55.174, R2: 0.425
Run ID: 952e219a87b64836a7219d02031b3d24, RMSE: 55.174, R2: 0.425
Run ID: e23b10833c3445a38780dd7cb4fc77b7, RMSE: 53.853, R2: 0.453
Run ID: 865213e88be240788f4a6ca9598b2c67, RMSE: 53.853, R2: 0.453
Run ID: c3eb76085a094a2fb9ccfaa9c0c155aa, RMSE: nan, R2: nan
Run ID: 5c706ce1b37e4dd5bac126c78311f799, RMSE: nan, R2: nan
Run ID: 99e1f6b2d4724a16991cc02f81b4def8, RMSE: nan, R2: nan
Run ID: 577c6399050546c88949833c33398041, RMSE: nan, R2: nan
Run ID: 5e2d4ff35fd9412b9a3f864b11a00fe6, RMSE: nan

In [ ]:
#          search_model_versions
versions = client.search_model_versions(
    filter_string = f"name='RandomForestRegressor' and run_id='{run.info.run_id}'")
print(f"\nModel versions for 'RandomForestRegressor'\n in run '{run.info.run_id}':")
print(versions)
for version in versions:
    print(
        "Version:", version.version,
        "Run ID:", version.run_id
    )
    
    
# Set an alias for the registered model 
# this make v3 is the champion model for the registered model "my_model"
client.set_registered_model_alias(
    "my_model",
    "champion",
    "3"
)


Model versions for 'RandomForestRegressor'
 in run '863466cafdcb4da7883bc7a01a9b3f65':
[]
